<h1 style="color:DodgerBlue">Индивидальный проект</h1>

## Название проекта

### Вариант № 18 — Система отзывов


## Описание проекта

Продолжение варианта № 18 «Система отзывов» из Sprint2.
Сохранены Review, ServiceReview, ProductReview, DigitalProductReview,
интерфейсы и прежние примеры взаимодействия объектов.

### Добавления текущего спринта

| Класс | Три новых свойства | Три новых метода |
|---|---|---|
| Review | Title, Language, IsEdited | EditText(string, string), DisplayReview(bool), MatchesRating(int) |
| ServiceReview | WaitMinutes, StaffName, Location | RateProduct(int), RateProduct(int, string), UpdateVisit(DateTime, string) |
| ProductReview | Category, WarrantyMonths, Seller | DisplayReview(), RateProduct(int, string), SetWarranty(int) |
| DigitalProductReview | FileFormat, SystemRequirements, UpdateNotes | DisplayReview(), UpdateVersion(string, string), IsCompatibleWith(string) |

**Полиморфизм:** вызовы DisplayReview через Review и через обобщённую коллекцию
используют реализации фактического типа объекта. В переопределениях вызывается base.

**Перегрузки:** вывод отзыва с параметром и без него, оценка с комментарием,
изменение версии с описанием обновления, вывод коллекции с фильтром и без него.

**Generic-класс:** ReviewCollection<T> where T : Review поддерживает добавление,
удаление, вывод, фильтрацию и средний рейтинг.
ReviewCollection<Review> хранит разные отзывы, а ReviewCollection<ProductReview> —
отзывы о товарах, включая цифровые.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

// Один класс может реализовать несколько интерфейсов.
public interface IVerifiable { void VerifyPurchase(); }
public interface IDiscountable { decimal CalculateDiscountPrice(decimal percent); }
public interface IRecommendable { void Recommend(bool recommended); }

// Базовый класс Review
public class Review
{

    // Новые свойства текущего спринта.
    public string Title { get; set; } = "Без заголовка";
    public string Language { get; set; } = "ru";
    public bool IsEdited { get; private set; }

    // Перегрузка редактирования: новый текст и заголовок.
    public void EditText(string text, string title)
    {
        EditText(text);
        Title = title;
        IsEdited = true;
    }
    // Перегрузка вывода вызывает виртуальный метод без параметров.
    public void DisplayReview(bool detailed)
    {
        if (detailed) Console.WriteLine(GetReviewDetails());
        else DisplayReview();
    }
    public bool MatchesRating(int minimum) { return Rating >= minimum; }

    public int ReviewId { get; set; }
    public string Text { get; set; }
    
private int rating;
    public int Rating
    {
        get { return rating; }
        protected set
        {
            if (value < 1 || value > 5)
                throw new ArgumentOutOfRangeException(nameof(Rating), "Рейтинг должен быть от 1 до 5.");
            rating = value;
        }
    }

    // Четыре новых свойства базового класса.
    public string Author { get; set; } = "Аноним";
    public DateTime CreatedAt { get; private set; } = DateTime.Now;
    public int HelpfulVotes { get; private set; }
    public bool IsPublished { get; private set; }

    // Четыре новых метода базового класса.
    public void EditText(string text)
    {
        if (string.IsNullOrWhiteSpace(text))
            throw new ArgumentException("Текст отзыва не должен быть пустым.", nameof(text));
        Text = text;
        IsEdited = true;
    }
    public void Publish() { IsPublished = true; }
    public void Unpublish() { IsPublished = false; }
    public void MarkHelpful() { HelpfulVotes++; }


    public Review(int reviewId, string text, int rating)
    {
        ReviewId = reviewId;
        Text = text;
        Rating = rating;
    }

    public virtual void DisplayReview()
    {
        Console.WriteLine($"Отзыв №{ReviewId}: {Text}. Рейтинг: {Rating}/5");
    }

    public virtual void RateProduct(int rating)
    {
        Rating = rating;
        Console.WriteLine($"Рейтингу присвоено значение: {Rating}/5");
    }

    public virtual string GetReviewDetails()
    {
        return $"ID отзыва: {ReviewId}; Текст: {Text}; Рейтинг: {Rating}/5";
    }

    // Взаимодействие двух объектов Review
    public void CompareRating(Review otherReview)
    {
        if (otherReview == null) throw new ArgumentNullException(nameof(otherReview));
        Console.WriteLine($"Сравнение отзывов №{ReviewId} и №{otherReview.ReviewId}:");

        if (Rating > otherReview.Rating)
            Console.WriteLine($"У отзыва №{ReviewId} рейтинг выше");
        else if (Rating < otherReview.Rating)
            Console.WriteLine($"У отзыва №{otherReview.ReviewId} рейтинг выше");
        else
            Console.WriteLine("Рейтинги одинаковые");
    }
}

// Производный класс для отзыва об услуге
public class ServiceReview : Review, IRecommendable, IDiscountable
{

    public int WaitMinutes { get; set; }
    public string StaffName { get; set; } = "Не указан";
    public string Location { get; private set; } = "Не указано";

    public override void RateProduct(int rating)
    {
        base.RateProduct(rating);
        Console.WriteLine($"Оценка услуги «{ServiceName}» обновлена.");
    }
    public void RateProduct(int rating, string comment)
    {
        if (string.IsNullOrWhiteSpace(comment)) throw new ArgumentException("Укажите комментарий.");
        RateProduct(rating);
        EditText(comment);
    }
    public void UpdateVisit(DateTime date, string location)
    {
        VisitDate = date;
        Location = location;
    }

    public DateTime VisitDate { get; set; }

    // Четыре новых свойства отзыва об услуге.
    public string ServiceName { get; private set; } = "Не указана";
    public string ProviderName { get; private set; } = "Не указан";
    public decimal Cost { get; private set; }
    public bool IsRecommended { get; private set; }

    // Четыре новых метода.
    public void UpdateServiceInfo(string serviceName, string providerName, decimal cost)
    {
        if (cost < 0) throw new ArgumentOutOfRangeException(nameof(cost));
        ServiceName = serviceName; ProviderName = providerName; Cost = cost;
    }
    public void Recommend(bool recommended) { IsRecommended = recommended; }
    public decimal CalculateDiscountPrice(decimal percent)
    {
        if (percent < 0 || percent > 100) throw new ArgumentOutOfRangeException(nameof(percent));
        return Cost * (1 - percent / 100m);
    }
    public int GetDaysSinceVisit(DateTime currentDate)
    {
        if (currentDate.Date < VisitDate.Date)
            throw new ArgumentException("Дата расчёта раньше посещения.", nameof(currentDate));
        return (currentDate.Date - VisitDate.Date).Days;
    }


    public ServiceReview(int reviewId, string text, int rating, DateTime visitDate)
        : base(reviewId, text, rating)
    {
        VisitDate = visitDate;
    }

    public override void DisplayReview()
    {
        Console.WriteLine(
            $"Отзыв об услуге №{ReviewId}: {Text}. " +
            $"Рейтинг: {Rating}/5. Дата посещения: {VisitDate:dd.MM.yyyy}");
    }

    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
               $"; Дата посещения: {VisitDate:dd.MM.yyyy}; услуга: {ServiceName}; " +
               $"исполнитель: {ProviderName}; стоимость: {Cost}; рекомендована: {IsRecommended}";
    }
}

// Производный класс для отзыва о товаре
public class ProductReview : Review, IVerifiable, IDiscountable
{

    public string Category { get; set; } = "Другое";
    public int WarrantyMonths { get; private set; }
    public string Seller { get; set; } = "Не указан";

    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Товар: {ProductName}; продавец: {Seller}; гарантия: {WarrantyMonths} мес.");
    }
    public void RateProduct(int rating, string comment)
    {
        if (string.IsNullOrWhiteSpace(comment)) throw new ArgumentException("Укажите комментарий.");
        RateProduct(rating);
        EditText(comment);
    }
    public void SetWarranty(int months)
    {
        if (months < 0) throw new ArgumentOutOfRangeException(nameof(months));
        WarrantyMonths = months;
    }

    public int ProductId { get; set; }

    // Четыре новых свойства отзыва о товаре.
    public string ProductName { get; private set; } = "Не указан";
    public string Brand { get; private set; } = "Не указан";
    public decimal PurchasePrice { get; private set; }
    public bool IsVerifiedPurchase { get; private set; }

    // Четыре новых метода.
    public void VerifyPurchase() { IsVerifiedPurchase = true; }
    public void UpdateProductInfo(string productName, string brand, decimal price)
    {
        if (price < 0) throw new ArgumentOutOfRangeException(nameof(price));
        ProductName = productName; Brand = brand; PurchasePrice = price;
    }
    public decimal CalculateDiscountPrice(decimal percent)
    {
        if (percent < 0 || percent > 100) throw new ArgumentOutOfRangeException(nameof(percent));
        return PurchasePrice * (1 - percent / 100m);
    }
    public string GetPurchaseSummary()
    {
        return $"{ProductName}, {Brand}; цена: {PurchasePrice}; подтверждена: {IsVerifiedPurchase}";
    }


    public ProductReview(int reviewId, string text, int rating, int productId)
        : base(reviewId, text, rating)
    {
        ProductId = productId;
    }

    public override void RateProduct(int rating)
    {
        Rating = rating;
        Console.WriteLine($"Товару с ID {ProductId} присвоен рейтинг: {Rating}/5");
    }

    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
               $"; ID продукта: {ProductId}; {GetPurchaseSummary()}";
    }
}

// Многоуровневая цепочка: Review -> ProductReview -> DigitalProductReview.
public class DigitalProductReview : ProductReview
{

    public string FileFormat { get; set; } = "exe";
    public string SystemRequirements { get; set; } = "Windows 10 и выше";
    public string UpdateNotes { get; private set; } = "Нет изменений";

    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Цифровой товар: {Platform}, версия {Version}, формат {FileFormat}");
    }
    public void UpdateVersion(string version, string notes)
    {
        UpdateVersion(version);
        UpdateNotes = notes;
    }
    public bool IsCompatibleWith(string platform)
    {
        return string.Equals(Platform, platform, StringComparison.OrdinalIgnoreCase);
    }

    public string Platform { get; set; }
    public string Version { get; private set; }
    public double DownloadSizeMb { get; private set; }
    public bool IsLicenseActivated { get; private set; }

    public DigitalProductReview(int reviewId, string text, int rating, int productId,
        string platform, string version, double downloadSizeMb)
        : base(reviewId, text, rating, productId)
    {
        if (double.IsNaN(downloadSizeMb) || double.IsInfinity(downloadSizeMb) || downloadSizeMb < 0)
            throw new ArgumentOutOfRangeException(nameof(downloadSizeMb));
        if (string.IsNullOrWhiteSpace(version)) throw new ArgumentException("Укажите версию.", nameof(version));
        Platform = platform; Version = version; DownloadSizeMb = downloadSizeMb;
    }

    public void ActivateLicense()
    {
        if (!IsVerifiedPurchase)
            throw new InvalidOperationException("Сначала подтвердите покупку.");
        IsLicenseActivated = true;
    }
    public void UpdateVersion(string version)
    {
        if (string.IsNullOrWhiteSpace(version)) throw new ArgumentException("Укажите версию.", nameof(version));
        Version = version;
    }
    public double GetDownloadSizeGb() { return DownloadSizeMb / 1000.0; }
    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
            $"; платформа: {Platform}; версия: {Version}; размер: {DownloadSizeMb} МБ; " +
            $"лицензия активирована: {IsLicenseActivated}";
    }
}


// Обобщённый класс: ограничение позволяет обращаться к методам Review.
public class ReviewCollection<T> where T : Review
{
    private readonly List<T> reviews = new List<T>();
    public int Count { get { return reviews.Count; } }
    public void Add(T review)
    {
        if (review == null) throw new ArgumentNullException(nameof(review));
        reviews.Add(review);
    }
    public bool Remove(T review) { return reviews.Remove(review); }
    public void DisplayAll()
    {
        foreach (T review in reviews) review.DisplayReview();
    }
    public void DisplayAll(int minimumRating)
    {
        foreach (T review in reviews)
            if (review.MatchesRating(minimumRating)) review.DisplayReview();
    }
    public double AverageRating()
    {
        if (reviews.Count == 0) return 0;
        double sum = 0;
        foreach (T review in reviews) sum += review.Rating;
        return sum / reviews.Count;
    }
}

// Пример использования
var serviceReview = new ServiceReview(
    1,
    "Хорошее обслуживание",
    5,
    new DateTime(2026, 9, 9));

var productReview = new ProductReview(
    2,
    "Качественный товар",
    4,
    101);

serviceReview.DisplayReview();
productReview.DisplayReview();

Console.WriteLine();
Console.WriteLine(serviceReview.GetReviewDetails());
Console.WriteLine(productReview.GetReviewDetails());

Console.WriteLine();
// Один объект взаимодействует с другим объектом
serviceReview.CompareRating(productReview);

Console.WriteLine();
productReview.RateProduct(5);
serviceReview.CompareRating(productReview);


Console.WriteLine("\nДополнительные свойства и методы:");
serviceReview.Author = "Дмитрий";
serviceReview.EditText("Вежливое обслуживание и быстрый ремонт");
serviceReview.UpdateServiceInfo("Ремонт ноутбука", "Сервис Плюс", 3000m);
serviceReview.Recommend(true);
serviceReview.Publish();
serviceReview.MarkHelpful();
Console.WriteLine(serviceReview.GetReviewDetails());
Console.WriteLine($"Автор: {serviceReview.Author}; создан: {serviceReview.CreatedAt:dd.MM.yyyy}; " +
    $"полезность: {serviceReview.HelpfulVotes}; опубликован: {serviceReview.IsPublished}");
Console.WriteLine($"Дней после посещения: {serviceReview.GetDaysSinceVisit(new DateTime(2026, 10, 20))}");
serviceReview.Unpublish();
Console.WriteLine($"После снятия с публикации: {serviceReview.IsPublished}");

productReview.UpdateProductInfo("Клавиатура", "Logitech", 5000m);
productReview.VerifyPurchase();
Console.WriteLine(productReview.GetPurchaseSummary());

var digitalReview = new DigitalProductReview(
    3, "Удобная программа", 4, 202, "Windows", "1.0", 1500);
digitalReview.UpdateProductInfo("Редактор фотографий", "PhotoSoft", 2000m);
digitalReview.Author = "Анна";
digitalReview.VerifyPurchase();
digitalReview.ActivateLicense();
digitalReview.UpdateVersion("1.1");
digitalReview.Publish();
Console.WriteLine($"Размер загрузки: {digitalReview.GetDownloadSizeGb()} ГБ");

// Полиморфизм: в массиве базового типа находятся разные производные объекты.
Review[] reviews = { serviceReview, productReview, digitalReview };
foreach (Review review in reviews)
    Console.WriteLine(review.GetReviewDetails());
digitalReview.CompareRating(productReview);

// Работа через интерфейсы: один объект поддерживает несколько контрактов.
IVerifiable verifiable = digitalReview;
verifiable.VerifyPurchase();
IRecommendable recommendable = serviceReview;
recommendable.Recommend(false);
Console.WriteLine($"Рекомендация после изменения: {serviceReview.IsRecommended}");
IDiscountable[] discountedItems = { serviceReview, productReview, digitalReview };
foreach (IDiscountable item in discountedItems)
    Console.WriteLine($"Стоимость со скидкой 10%: {item.CalculateDiscountPrice(10)}");

// Ошибочные данные отклоняются, а прежний рейтинг сохраняется.
try { productReview.RateProduct(6); }
catch (ArgumentOutOfRangeException) { Console.WriteLine("Рейтинг вне диапазона 1–5 отклонён."); }
Console.WriteLine($"Рейтинг после проверки: {productReview.Rating}/5");


Console.WriteLine("\nНовый спринт: полиморфизм, перегрузки и generics");
serviceReview.WaitMinutes = 10;
serviceReview.StaffName = "Иван";
serviceReview.UpdateVisit(new DateTime(2026, 10, 10), "Тюмень");
serviceReview.RateProduct(4, "Работа выполнена хорошо");
serviceReview.EditText("Хороший ремонт", "Отзыв о сервисе");
Console.WriteLine($"Заголовок: {serviceReview.Title}; язык: {serviceReview.Language}; изменён: {serviceReview.IsEdited}");
Console.WriteLine($"Ожидание: {serviceReview.WaitMinutes}; сотрудник: {serviceReview.StaffName}; место: {serviceReview.Location}");
productReview.Category = "Аксессуары";
productReview.Seller = "Магазин техники";
productReview.SetWarranty(24);
productReview.RateProduct(5, "Отличная клавиатура");
Console.WriteLine($"Категория: {productReview.Category}");
digitalReview.UpdateVersion("2.0", "Добавлены новые фильтры");
Console.WriteLine($"Требования: {digitalReview.SystemRequirements}; изменения: {digitalReview.UpdateNotes}");
Console.WriteLine($"Совместимость с Windows: {digitalReview.IsCompatibleWith("windows")}");

// Одинаковый вызов через Review вызывает разные реализации.
foreach (Review item in reviews) item.DisplayReview();
productReview.DisplayReview(true);
digitalReview.DisplayReview(false);

ReviewCollection<Review> allReviews = new ReviewCollection<Review>();
allReviews.Add(serviceReview);
allReviews.Add(productReview);
allReviews.Add(digitalReview);
allReviews.DisplayAll();
Console.WriteLine($"Средний рейтинг: {allReviews.AverageRating():F2}");
Console.WriteLine("Только отзывы с рейтингом от 5:");
allReviews.DisplayAll(5);
ReviewCollection<ProductReview> products = new ReviewCollection<ProductReview>();
products.Add(productReview);
products.Add(digitalReview); // Подкласс ProductReview разрешён.
// products.Add(serviceReview); // Ошибка компиляции: это не ProductReview.
products.DisplayAll();
Console.WriteLine($"Удаление цифрового отзыва: {products.Remove(digitalReview)}");
Console.WriteLine($"Осталось отзывов о товарах: {products.Count}");


### Пояснение

Сохранены классы и методы прошлого спринта. Добавлено по три свойства и три метода в каждый существующий класс (включая переопределения). Переопределения используют `override`, перегрузки имеют разные параметры. `ReviewCollection<T>` работает с отзывами без приведения к `object`. Для пустой коллекции `AverageRating()` возвращает 0 по принятому в проекте соглашению.

Выполните ячейку в отдельном ядре `.NET (C#)`.